# Week 3 — Embeddings, UMAP, Clustering
**Hùng**: Doc2Vec embeddings + DBSCAN clustering  
**Bảo**: UMAP dimensionality reduction

In [4]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "gensim", "umap-learn", "scikit-learn", "polars", "pyarrow", "tqdm", "matplotlib", "seaborn"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'gensim', 'umap-learn', 'scikit-learn', 'polars', 'pyarrow', 'tqdm', 'matplotlib', 'seaborn'], returncode=0)

In [7]:
import polars as pl
import numpy as np
import re
from pathlib import Path
from tqdm import tqdm

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.utils import simple_preprocess

import umap
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize

import matplotlib.pyplot as plt
import seaborn as sns

PARQUET = Path(r"F:\UK-Russia\jupyter\output\tweets_clean.parquet")
OUT_DIR = Path(r"F:\UK-Russia\jupyter\output")
OUT_DIR.mkdir(exist_ok=True)

## 1. Load data

In [9]:
df = pl.read_parquet(str(PARQUET / "*.parquet") if PARQUET.is_dir() else PARQUET, columns=["tweetid", "text"])
print(f"{len(df):,} tweets")
df.head(3)


FileNotFoundError: No such file or directory (os error 2): F:\UK-Russia\jupyter\output\tweets_clean.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'select'
	[3] 'sink'


## 2. Doc2Vec embeddings

In [ ]:
def clean(text):
    text = re.sub(r"http\S+", "", text)          # bỏ link
    text = re.sub(r"@\w+", "", text)             # bỏ mention
    text = re.sub(r"[^\w\s]", "", text.lower())  # lowercase, bỏ ký tự đặc biệt
    return text.strip()

texts = df["text"].to_list()
tweetids = df["tweetid"].to_list()

corpus = [
    TaggedDocument(words=simple_preprocess(clean(t)), tags=[i])
    for i, t in enumerate(tqdm(texts, desc="tokenizing"))
    if t and len(t.strip()) > 5
]
print(f"{len(corpus):,} documents")

tokenizing:  13%|█▎        | 1492415/11099751 [00:54<04:20, 36858.38it/s]

KeyboardInterrupt: 

In [ ]:
model = Doc2Vec(
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    epochs=10,
    dm=1,
)
model.build_vocab(corpus)
model.train(corpus, total_examples=model.corpus_count, epochs=model.epochs)

model.save(str(OUT_DIR / "doc2vec.model"))
print("saved doc2vec.model")

# lấy vector cho từng doc
valid_ids = [doc.tags[0] for doc in corpus]
vectors = np.array([model.dv[i] for i in valid_ids])
print(f"vectors shape: {vectors.shape}")

## 3. UMAP — giảm chiều 
- 100D → 10D để clustering
- 100D → 2D để visualize

In [ ]:
vecs_norm = normalize(vectors)  # cosine-friendly

# 100D -> 10D cho DBSCAN
reducer_10d = umap.UMAP(n_components=10, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=42)
emb_10d = reducer_10d.fit_transform(vecs_norm)

# 100D -> 2D để vẽ
reducer_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
emb_2d = reducer_2d.fit_transform(vecs_norm)

print(f"10D: {emb_10d.shape}  |  2D: {emb_2d.shape}")

## 4. DBSCAN clustering 

In [ ]:
dbscan = DBSCAN(eps=0.3, min_samples=10, metric="euclidean", n_jobs=-1)
labels = dbscan.fit_predict(emb_10d)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_pct = (labels == -1).sum() / len(labels) * 100

print(f"clusters   : {n_clusters}")
print(f"noise      : {noise_pct:.1f}%")
print(f"largest    : {np.bincount(labels[labels >= 0]).max():,} tweets")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 7))

mask_noise = labels == -1
ax.scatter(emb_2d[mask_noise, 0], emb_2d[mask_noise, 1],
           c="lightgray", s=1, alpha=0.3, label="noise")

palette = sns.color_palette("tab20", n_colors=min(n_clusters, 20))
for cid in range(min(n_clusters, 20)):
    mask = labels == cid
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
               c=[palette[cid]], s=2, alpha=0.6, label=f"c{cid}")

ax.set_title(f"UMAP 2D — {n_clusters} clusters (DBSCAN)")
ax.legend(markerscale=5, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "umap_clusters.png"), dpi=150)
plt.show()
print("saved umap_clusters.png")

In [ ]:
# lưu kết quả để dùng ở week 4
result = pl.DataFrame({
    "tweetid": [tweetids[i] for i in valid_ids],
    "cluster": labels.tolist(),
    "umap_x":  emb_2d[:, 0].tolist(),
    "umap_y":  emb_2d[:, 1].tolist(),
})
result.write_parquet(str(OUT_DIR / "clusters.parquet"))
print(f"saved clusters.parquet — {len(result):,} rows")
result.head(5)